# 🏎️ F1 Season Monte Carlo Simulator — Results Visualization

This notebook loads the output CSVs from the Monte Carlo simulation and visualizes the probability distributions for each driver across all championship finishing positions.

Each CSV represents 10,000+ simulated seasons. Values in each cell are the **probability** (0–1) that a given driver finishes the championship in that position.

---

In [ ]:
%pip install pandas
%pip install numpy
%pip install matplotlib
%pip install seaborn
%pip install pathlib

## 0. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path

# UPDATE TO UPCOMING RACE
next_race="Monaco Grand Prix"

# UPDATE TO pre OR post IN REFERENCE TO QUALUIFYING FOR ABOVE GRAND PRIX
pre_post="pre"

OUTPUT_DIR = Path("..") / f"csv_files/{next_race}"
GRAPH_DIR = Path("..") / f"graph_files/{next_race}/{pre_post}"

# Plot styling
plt.rcParams['figure.facecolor'] = '#0f0f0f'
plt.rcParams['axes.facecolor'] = '#1a1a1a'
plt.rcParams['axes.edgecolor'] = '#333333'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = 'white'
plt.rcParams['xtick.color'] = 'white'
plt.rcParams['ytick.color'] = 'white'
plt.rcParams['grid.color'] = '#2a2a2a'
plt.rcParams['font.family'] = 'monospace'
plt.rcParams['figure.dpi'] = 120

F1_RED = '#E8002D'
F1_SILVER = '#C0C0C0'

print('Setup complete.')

## 1. Load Data

In [ ]:
def load_sim_csv(filename):
    """Load a simulation output CSV. Index = driver codes, columns = finishing positions + DNF."""
    df = pd.read_csv(OUTPUT_DIR / filename, index_col=0)
    # Ensure column names are strings for consistent handling
    df.columns = df.columns.astype(str)
    return df

# Load both outputs — update filenames if yours differ
race_prob = load_sim_csv(f"{pre_post}_q_race.csv")
season_prob = load_sim_csv(f"{pre_post}_q_season.csv")

print(f"Loaded race_prob:       {race_prob.shape[0]} drivers x {race_prob.shape[1]} positions")
print(f"Loaded season_prob:  {season_prob.shape[0]} drivers x {season_prob.shape[1]} positions")
print(f"\nDrivers: {list(race_prob.index)}")

## 2. Quick Sanity Check

In [ ]:
# Each driver's probabilities should sum to ~1.0
row_sums = race_prob.sum(axis=1)
print("Row sums (should be ~1.0 for each driver):")
print(row_sums.round(4).to_string())

# Each position column should also sum to ~1.0 (excluding DNF)
numeric_cols = [c for c in season_prob.columns if c != 'DNF']
col_sums = season_prob[numeric_cols].sum(axis=0)
print(f"\nColumn sums (positions 1–{len(numeric_cols)}, should each be ~1.0):")
print(col_sums.round(4).to_string())

## 3. Championship Win Probability — Bar Chart

P(driver finishes P1 in the championship) for all drivers, sorted descending.

In [ ]:
win_probs = race_prob['1'].sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))

colors = [F1_RED if i == 0 else F1_SILVER for i in range(len(win_probs))]
bars = ax.bar(win_probs.index, win_probs.values * 100, color=colors, edgecolor='none', width=0.65)

# Value labels on bars
for bar, val in zip(bars, win_probs.values):
    if val >= 0.01:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.4,
            f'{val*100:.1f}%',
            ha='center', va='bottom', fontsize=8, color='white'
        )

ax.set_title('Race Win Probability — Next Race (10,000 simulations)', 
             fontsize=13, pad=15, color='white')
ax.set_ylabel('Probability (%)', fontsize=10)
ax.set_xlabel('Driver', fontsize=10)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.grid(axis='y', linewidth=0.5)
ax.set_axisbelow(True)

png_location='race_win_probability.png'

GRAPH_DIR.mkdir(parents=True, exist_ok=True)

plt.tight_layout()
plt.savefig(GRAPH_DIR / png_location, dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print('Saved: graph_files/'+png_location)

season_win_probs = season_prob['1'].sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))

colors = [F1_RED if i == 0 else F1_SILVER for i in range(len(season_win_probs))]
bars = ax.bar(season_win_probs.index, season_win_probs.values * 100, color=colors, edgecolor='none', width=0.65)

# Value labels on bars
for bar, val in zip(bars, season_win_probs.values):
    if val >= 0.01:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.4,
            f'{val*100:.1f}%',
            ha='center', va='bottom', fontsize=8, color='white'
        )

ax.set_title('Championship Win Probability — Full Season (10,000 simulations)', 
             fontsize=13, pad=15, color='white')
ax.set_ylabel('Probability (%)', fontsize=10)
ax.set_xlabel('Driver', fontsize=10)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.grid(axis='y', linewidth=0.5)
ax.set_axisbelow(True)

png_location='wdc_win_probability.png'

plt.tight_layout()
plt.savefig(GRAPH_DIR / png_location, dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print('Saved: graph_files/'+png_location)

## 4. Full Probability Heatmap — All Drivers, All Positions

Drivers sorted by expected finishing position (probability-weighted mean). Higher probability = darker red.

In [ ]:
# Sort drivers by probability-weighted expected position
#Next Race
numeric_cols = [c for c in race_prob.columns if c.isdigit()]
positions = np.array([int(c) for c in numeric_cols])

expected_pos = race_prob[numeric_cols].apply(
    lambda row: np.dot(row.values, positions), axis=1
)
sorted_drivers = expected_pos.sort_values().index
heatmap_data = race_prob.loc[sorted_drivers, numeric_cols]

fig, ax = plt.subplots(figsize=(18, 10))

sns.heatmap(
    heatmap_data,
    ax=ax,
    cmap=sns.color_palette('rocket_r', as_cmap=True),
    linewidths=0.3,
    linecolor='#0f0f0f',
    cbar_kws={'label': 'Probability', 'shrink': 0.6},
    fmt='',
    annot=heatmap_data.map(lambda x: f'{x*100:.0f}%' if x >= 0.04 else ''),
    annot_kws={'size': 7, 'color': 'white'},
    vmin=0,
    vmax=heatmap_data.values.max()
)

ax.set_title(next_race+' Race Win Probability — Race (10,000 simulations)',
             fontsize=13, pad=15, color='white')
ax.set_xlabel('Championship Position', fontsize=10)
ax.set_ylabel('Driver', fontsize=10)
ax.tick_params(axis='both', labelsize=9)

# Colorbar label color
ax.collections[0].colorbar.ax.yaxis.label.set_color('white')
ax.collections[0].colorbar.ax.tick_params(colors='white')

png_location='race_full_heatmap.png'

plt.tight_layout()
plt.savefig(GRAPH_DIR / png_location, dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print('Saved: graph_files/'+png_location)

# Sort drivers by probability-weighted expected position
#Season
numeric_cols = [c for c in season_prob.columns if c.isdigit()]
positions = np.array([int(c) for c in numeric_cols])

expected_pos = season_prob[numeric_cols].apply(
    lambda row: np.dot(row.values, positions), axis=1
)
sorted_drivers = expected_pos.sort_values().index
heatmap_data = season_prob.loc[sorted_drivers, numeric_cols]

fig, ax = plt.subplots(figsize=(18, 10))

sns.heatmap(
    heatmap_data,
    ax=ax,
    cmap=sns.color_palette('rocket_r', as_cmap=True),
    linewidths=0.3,
    linecolor='#0f0f0f',
    cbar_kws={'label': 'Probability', 'shrink': 0.6},
    fmt='',
    annot=heatmap_data.map(lambda x: f'{x*100:.0f}%' if x >= 0.04 else ''),
    annot_kws={'size': 7, 'color': 'white'},
    vmin=0,
    vmax=heatmap_data.values.max()
)

ax.set_title('Season Win Probability —'+next_race+' Next Race (10,000 simulations)',
             fontsize=13, pad=15, color='white')
ax.set_xlabel('Championship Position', fontsize=10)
ax.set_ylabel('Driver', fontsize=10)
ax.tick_params(axis='both', labelsize=9)

# Colorbar label color
ax.collections[0].colorbar.ax.yaxis.label.set_color('white')
ax.collections[0].colorbar.ax.tick_params(colors='white')

png_location='season_full_heatmap.png'

plt.tight_layout()
plt.savefig(GRAPH_DIR / png_location, dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print('Saved: graph_files/'+png_location)

## 5. Podium Probability — Top 3 Finish

P(driver finishes P1, P2, or P3 in the next race), sorted by total podium probability.

In [ ]:
podium_prob = (race_prob['1'] + race_prob['2'] + race_prob['3']).sort_values(ascending=True)

# Stacked bar: P1 / P2 / P3 contributions
p1 = race_prob.loc[podium_prob.index, '1']
p2 = race_prob.loc[podium_prob.index, '2']
p3 = race_prob.loc[podium_prob.index, '3']

fig, ax = plt.subplots(figsize=(10, 9))

ax.barh(podium_prob.index, p1 * 100, color=F1_RED,    label='P1 — Champion',  height=0.6)
ax.barh(podium_prob.index, p2 * 100, left=p1 * 100,   color='#C0C0C0', label='P2',           height=0.6)
ax.barh(podium_prob.index, p3 * 100, left=(p1+p2)*100, color='#CD7F32', label='P3',           height=0.6)

# Total label at end of each bar
for driver in podium_prob.index:
    total = podium_prob[driver] * 100
    ax.text(total + 0.5, driver, f'{total:.1f}%', va='center', fontsize=8, color='white')

ax.set_title('Podium Finish Probability (P1 + P2 + P3)\nStacked by position contribution',
             fontsize=13, pad=15, color='white')
ax.set_xlabel('Probability (%)', fontsize=10)
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.legend(loc='lower right', framealpha=0.2, fontsize=9)
ax.grid(axis='x', linewidth=0.5)
ax.set_axisbelow(True)

plt_save='podium_probability.png'

plt.tight_layout()
plt.savefig(GRAPH_DIR / plt_save, dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print('Saved: graph_files/'+plt_save)

## 6. Individual Driver Deep Dive

Distribution of finishing positions for a single driver. Change `DRIVER` to any driver code in your dataset.

In [ ]:
DRIVER = 'LEC'  # <-- change this to any driver code

def driver_bar_chart(driver,prob,length):
    if driver not in prob.index:
        print(f"Driver '{driver}' not found. Available: {list(prob.index)}")
    else:
        driver_data = prob.loc[driver]
        numeric_data = driver_data[[c for c in driver_data.index if c.isdigit()]]
        dnf_prob = driver_data.get('DNF', 0)

        exp_pos = np.dot(numeric_data.values, [int(c) for c in numeric_data.index])

        fig, ax = plt.subplots(figsize=(13, 5))

        bar_colors = [F1_RED if p == numeric_data.idxmax() else '#444444' for p in numeric_data.index]
        bars = ax.bar(numeric_data.index, numeric_data.values * 100,
                    color=bar_colors, edgecolor='none', width=0.7)

        for bar, val in zip(bars, numeric_data.values):
            if val >= 0.01:
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                        f'{val*100:.1f}%', ha='center', va='bottom', fontsize=7.5, color='white')

        ax.axvline(x=str(int(round(exp_pos))), color=F1_SILVER, linestyle='--',
                linewidth=1.2, label=f'Expected position: {exp_pos:.1f}')
        
        if(length=="r"):
            ax.set_title(f'{driver} — {next_race} Position Probability Distribution\n'
                 f'DNF probability: {dnf_prob*100:.1f}%  |  Expected finish: P{exp_pos:.1f}',
                 fontsize=12, pad=12, color='white')
            ax.set_xlabel('Race Position', fontsize=10)
            save_loc=f'r{driver}_distribution.png'
        else:
            ax.set_title(f'{driver} — PRE {next_race} Championship Position Probability Distribution\n'
                        f'Expected finish: P{exp_pos:.1f}',
                        fontsize=12, pad=12, color='white')
            ax.set_xlabel('Championship Position', fontsize=10)
            save_loc=f'c{driver}_distribution.png'
        ax.set_ylabel('Probability (%)', fontsize=10)
        ax.yaxis.set_major_formatter(mtick.PercentFormatter())
        ax.legend(fontsize=9, framealpha=0.2)
        ax.grid(axis='y', linewidth=0.5)
        ax.set_axisbelow(True)

        plt.tight_layout()

        FOLDER_PATH = Path(GRAPH_DIR / driver)

        # Create the folder safely
        FOLDER_PATH.mkdir(parents=True, exist_ok=True)

        plt.savefig(FOLDER_PATH/save_loc, dpi=150,
                    bbox_inches='tight', facecolor='#0f0f0f')
        print(f'Saved: graph_files/{driver}/{length}{driver}_distribution.png')

for driver in race_prob.index:
    driver_bar_chart(driver,race_prob,"r")
    driver_bar_chart(driver,season_prob,"c")

## 7. Points Standings Comparison — Forecast vs Current

Compares the two simulation outputs side by side. Useful when one CSV is a mid-season snapshot and the other is end-of-season forecast.

In [ ]:
# Race win probability vs championship probability scatter
race_p1 = race_prob['1']
season_p1 = season_prob['1']

fig, ax = plt.subplots(figsize=(9, 7))

# Plot each driver as a dot
for driver in race_p1.index:
    x = race_p1[driver] * 100
    y = season_p1[driver] * 100
    ax.scatter(x, y, color=F1_RED, s=60, zorder=3)
    ax.annotate(driver, (x, y), textcoords="offset points",
                xytext=(6, 4), fontsize=8, color='white')

# Reference line — if race dominance = season dominance perfectly, points fall on this line
max_val = max(race_p1.max(), season_p1.max()) * 100 * 1.1
ax.plot([0, max_val], [0, max_val], color='#444444', linestyle='--',
        linewidth=1, label='Race win % = Championship win %')

ax.set_xlabel('Race Win Probability — Next Race (%)', fontsize=10)
ax.set_ylabel('Championship Win Probability — Season Forecast (%)', fontsize=10)
ax.set_title('Race Win vs Championship Win Probability\nDrivers above the line are stronger over a season than a single race',
             fontsize=11, pad=14, color='white')
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.legend(fontsize=8, framealpha=0.2)
ax.grid(linewidth=0.4)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig(GRAPH_DIR / 'race_vs_season.png', dpi=150,
            bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print('Saved: graph_files/race_vs_season.png')

---

## Summary

| Chart | File | Description |
|---|---|---|
| Race vs Season Scatter | `race_vs_season.png` | Race win % vs championship win % per driver |

All charts are saved to `output/` and can be used directly in LinkedIn posts or reports.